In [ ]:
# ============================================================
# SKETCHBYTE — CELL 4
# QWEN3-TTS 1.7B VOICE GENERATION (STREAMING AUDIO CHUNKS)
# ============================================================

import os
import gc
import time
import sys
import re
import torch
import numpy as np
import soundfile as sf

from IPython.display import Audio, display
from ipywidgets import Output
from google.colab import files
from qwen_tts import Qwen3TTSModel

print("=" * 70)
print("SKETCHBYTE — CELL 4 (VRAM OPTIMIZED + LIVE AUDIO PLAYERS)")
print("QWEN3-TTS 1.7B VOICE CLONING")
print("=" * 70)

# ============================================================
# 1. CHECK PREVIOUS STEPS
# ============================================================

if "formatted_script" not in globals() or not formatted_script:
    script_file = "/content/SketchByte_TTS_Ready_Script.txt"
    if os.path.isfile(script_file):
        with open(script_file, "r", encoding="utf-8") as f:
            formatted_script = f.read().strip()
    if not formatted_script:
        raise RuntimeError("\n❌ TTS-ready script not found. Run Cell 3 first.")

if "REFERENCE_AUDIO" not in globals() or "REFERENCE_TEXT" not in globals():
    raise RuntimeError("\n❌ Reference voice/text missing. Run Cells 1 & 2 first.")

# ============================================================
# 2. UNLOAD PREVIOUS MODELS & FREE VRAM
# ============================================================

print("\n[1/5] Freeing GPU memory...")
print("-" * 70)

for var in ['formatter_model', 'formatter_tokenizer']:
    if var in globals():
        del globals()[var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("✅ VRAM Cleared.")

# ============================================================
# 3. LOAD QWEN3-TTS 1.7B (LOW VRAM SETTINGS)
# ============================================================

print("\n[2/5] Loading Qwen3-TTS 1.7B...")
print("-" * 70)

try:
    # Using low_cpu_mem_usage and sdpa for optimized VRAM footprint
    tts_model = Qwen3TTSModel.from_pretrained(
        TTS_MODEL,
        device_map="cuda:0",
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )
except Exception as e:
    raise RuntimeError(f"\n❌ Failed to load TTS model: {e}")

print("✅ TTS Model Loaded.")

# ============================================================
# 4. CHUNK-BASED GENERATION WITH LIVE AUDIO STREAMING PLAYER
# ============================================================

def split_script(text, max_words=200):
    # Split on sentence boundaries so chunks never slice mid-sentence.
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    current = []
    current_len = 0
    for sentence in sentences:
        words = sentence.split()
        if current_len + len(words) > max_words and current:
            yield " ".join(current)
            current = []
            current_len = 0
        current.extend(words)
        current_len += len(words)
    if current:
        yield " ".join(current)

print("\n[3/5] Starting Chunked Generation...")
print("-" * 70)

full_words = formatted_script.split()
total_words = len(full_words)
chunks = list(split_script(formatted_script))
total_chunks = len(chunks)

print(f"📝 Total words: {total_words:,}")
print(f"📦 Total chunks: {total_chunks}")
print("\n🎙️ Starting Qwen3-TTS generation...")

# Prepare dynamic widget layout to render audio players live as they finish
live_audio_output = Output()
display(live_audio_output)

# Build the voice-clone prompt ONCE so the reference audio is not
# re-encoded for every chunk (saves time and VRAM spikes).
print("\n🎤 Preparing permanent voice reference (once)...")
voice_prompt = tts_model.create_voice_clone_prompt(
    ref_audio=REFERENCE_AUDIO,
    ref_text=REFERENCE_TEXT,
)
print("✅ Reference voice ready.")

all_audio = []
generated_word_count = 0
start_time = time.time()
final_sr = 24000

try:
    # Use inference_mode to avoid storing tracking gradients (saves massive VRAM)
    with torch.inference_mode():
        for i, chunk_text in enumerate(chunks):
            chunk_word_count = len(chunk_text.split())
            current_chunk_idx = i + 1

            # Progress Bar Display
            percent = int((current_chunk_idx / total_chunks) * 100)
            bar = "█" * (percent // 5) + "░" * (20 - (percent // 5))
            elapsed = time.time() - start_time

            sys.stdout.write(
                f"\rProgress: [{bar}] {percent}% | "
                f"Words: {generated_word_count}/{total_words} | "
                f"Chunk: {current_chunk_idx}/{total_chunks} | "
                f"Elapsed: {int(elapsed // 60):02d}:{int(elapsed % 60):02d}"
            )
            sys.stdout.flush()

            # Generate Audio for Chunk (uses the cached reference prompt)
            wavs, sr = tts_model.generate_voice_clone(
                text=chunk_text,
                language="English",
                voice_clone_prompt=voice_prompt,
            )

            chunk_audio = wavs[0]
            all_audio.append(chunk_audio)
            final_sr = sr
            generated_word_count += chunk_word_count

            # Stream the playable audio chunk dynamically directly under the progress bar
            with live_audio_output:
                print(f"🔊 Chunk {current_chunk_idx}/{total_chunks} Completed:")
                display(Audio(chunk_audio, rate=sr, autoplay=False))

            # Clean up intermediate tensors and release VRAM cache back to GPU actively
            del wavs
            gc.collect()
            torch.cuda.empty_cache()

    # Final update
    sys.stdout.write(f"\rProgress: [{'█'*20}] 100% | Words: {total_words}/{total_words} | DONE!                      \n")

except Exception as e:
    print(f"\n\n❌ Generation failed at chunk {i+1}: {e}")
    raise e

# ============================================================
# 5. CONCATENATE & SAVE
# ============================================================

print("\n[4/5] Merging and saving final audio...")
print("-" * 70)
if not all_audio:
    raise RuntimeError("\n❌ No audio chunks generated. Check your script text.")

final_wav = np.concatenate(all_audio, axis=0)
output_path = "/content/SketchByte_Voiceover.wav"
sf.write(output_path, final_wav, final_sr)

print("\n🎧 Complete Final Narration Playback:")
display(Audio(output_path, autoplay=False))
files.download(output_path)

print("\n" + "=" * 70)
print("🟢 SKETCHBYTE VOICEOVER COMPLETE")
print("=" * 70)
print(f"✅ Final Duration: {len(final_wav)/final_sr/60:.2f} minutes")
print(f"✅ Saved to: {output_path}")